Настоящий раздел содержит результаты выполнения технического задания (ТЗ) в контексте игровой индустрии. В качестве инструментария выбран язык запросов SQL, обеспечивающий эффективное взаимодействие с реляционными базами данных. Практическая реализация, включая загрузку данных и выполнение SQL‑запросов, выполнена в интегрированной среде разработки pgAdmin 4. <br>

**Структура файла**

1. Таблица players (игроки) <br>
Хранит аккаунты пользователей.<br>
<br>
Поля:<br>
player_id (PK) — уникальный ID игрока<br>
username — никнейм<br>
email — почта<br>
created_at — дата регистрации<br>
level — уровень аккаунта<br>
<br>
Назначение:<br>
Основная таблица пользователей игры.<br>

2. Таблица characters (персонажи)<br>
Игровые персонажи, принадлежащие игрокам.<br>
<br>
Поля:<br>
character_id — ID персонажа<br>
player_id (PK)  — владелец<br>
name — имя персонажа<br>
class — класс (воин, маг и т.д.)<br>

3. Таблица items (предметы)<br>
Все игровые предметы.<br>
<br>
Поля:<br>
item_id (PK) — ID предмета<br>
name — название<br>
type — тип (оружие, броня, зелье)<br>
rarity — редкость<br>
value — стоимость<br>
<br>
Назначение:<br>
Каталог всех предметов в игре.<br>

4. Таблица inventory (инвентарь)<br>
Связь персонажей и предметов (многие-ко-многим).<br>
<br>
Поля:<br>
inventory_id (PK) — ID записи<br>
character_id (FK → characters.character_id) — персонаж<br>
item_id (FK → items.item_id) — предмет<br>
quantity — количество<br>
<br>
Назначение:<br>
Показывает, какие предметы есть у персонажа.<br>

5. Таблица quests (квесты)<br>
Игровые задания.<br>
<br>
Поля:<br>
quest_id (PK) — ID квеста<br>
title — название<br>
reward_gold — награда (золото)<br>
reward_exp — опыт<br>
<br>
Назначение:<br>
Список всех доступных квестов.<br>


**Создание таблиц в pgAdmin 4**<br>
<br>
Таблица №1<br>

```sql
CREATE TABLE players(
	player_id INTEGER PRIMARY KEY,
	username VARCHAR(100) NOT NULL,
	email VARCHAR(150),
	created_at DATE NOT NULL,
	level INTEGER
)

Таблица №2

CREATE TABLE characters(
	character_id INTEGER,
	player_id INTEGER PRIMARY KEY,
	name VARCHAR(150),
	class VARCHAR(50)
)

Таблица №3

CREATE TABLE inventory(
	inventory_id INTEGER PRIMARY KEY,
	player_id INTEGER,
	item_id INTEGER,
	quantity INTEGER
)


Таблица №4

CREATE TABLE items(
	item_id INTEGER PRIMARY KEY,
	name VARCHAR(150),
	type VARCHAR(100),
	rarity VARCHAR(100),
	value INTEGER
)

Таблица №5

CREATE TABLE quests(
	quest_id INTEGER PRIMARY KEY,
	title VARCHAR(300),
	reward_gold INTEGER,
	reward_exp INTEGER
)

**Задачи:**

🟢**1. БАЗА (SELECT, WHERE)**

Задача 1
Вывести всех игроков, зарегистрированных за последний месяц.

```sql
select * 
from players
WHERE created_at >= DATE_TRUNC('month', NOW())

Задача 2
Найти игроков выше 20 уровня.

select * 
from players
where level >= '20'

Задача 3
Вывести всех персонажей класса "маг".

select name
from characters
where class = 'Маг'


Задача 4
Найти предметы редкости "легендарный".

select name
from items
where rarity = 'легендарный'

Задача 5
Вывести персонажей, созданных в январе 2026.

select username
from players
WHERE EXTRACT(MONTH FROM created_at) = 1
  AND EXTRACT(YEAR FROM created_at) = 2026;


🟡 **2. АГРЕГАЦИИ**

```sql
Задача 6
Посчитать количество игроков.

select count(*)
from players

Задача 7
Найти средний уровень игроков.

select avg(level)
from players

Задача 8
Посчитать средний уровень по каждому месяцу.

select DATE_TRUNC('month', created_at) as month, avg(level)
from players
group by DATE_TRUNC('month', created_at)
order by DATE_TRUNC('month', created_at) asc


Задача 9
Посчитать количество предметов каждого типа.

select type, count(type)
from items
group by type

Задача 10
Найти максимальный уровень персонажа в игре.

SELECT *
FROM players
WHERE level = (SELECT MAX(level) FROM players);

🟠 **3. JOIN**

```sql
Задача 11
Вывести игроков и их персонажей.

select username, class
from players as t1
left join characters as t2
on t1.player_id = t2.player_id


Задача 12
Показать инвентарь каждого персонажа (предмет + количество).

select username, name, quantity
from players as t1
left join inventory as t2
on t1.player_id = t2.player_id
left join items as t3
on t2.item_id = t3.item_id


Задача 13
Вывести username и имя персонажа.

select username, name
from players as t1
left join characters as t2
on t1.player_id = t2.player_id

Задача 14
Найти персонажей без предметов.

select *
from players as t1
left join inventory as t2
on t1.player_id = t2.player_id
where inventory_id is null

Задача 15
Найти игроков без персонажей.
select *
from players as t1
left join characters as t2
on t1.player_id = t2.player_id
where name is null

🔵 **4. ПРОДВИНУТЫЕ JOIN + АНАЛИТИКА**

```sql

Задача 16
Посчитать общее количество предметов у каждого пользователя.

select username, sum(quantity) as sum_quantity
from players as t1
left join inventory as t2
on t1.player_id = t2.player_id
left join items as t3
on t2.item_id = t3.item_id
group by username

Задача 17
Найти топ-5 самых "богатых" персонажей (по сумме value предметов).

select username, sum(value) as top_value
from players as t1
left join inventory as t2
on t1.player_id = t2.player_id
left join items as t3
on t2.item_id = t3.item_id
where value is not null
group by username
order by top_value desc
limit 5

Задача 18
Посчитать общую стоимость инвентаря каждого игрока.

select username, sum(value) as top_value
from players as t1
left join inventory as t2
on t1.player_id = t2.player_id
left join items as t3
on t2.item_id = t3.item_id
where value is not null
group by username
order by top_value desc


Задача 19
Найти самый популярный предмет (чаще всего встречается). 

select COALESCE(name, 'Отсутствуют данные') as name_new, count(*)
from players as t1
left join inventory as t2
on t1.player_id = t2.player_id
left join items as t3
on t2.item_id = t3.item_id
group by name_new
limit 1



Задача 20
Найти самый редкий используемый предмет.

select *
from (select name, count(name)
from players as t1
left join inventory as t2
on t1.player_id = t2.player_id
left join items as t3
on t2.item_id = t3.item_id
where name is not null
group by name)
order by count
limit 1

🟣 **5. ПОДЗАПРОСЫ**

```sql

Задача 21
Найти персонажей с уровнем выше среднего.

select *
from players
where level >= (select avg(level) from players)
order by level

Задача 22
Найти предметы дороже среднего значения.

select name, value
from items
where value >= (select avg(value) from items)

🔴 **6. ОКОННЫЕ ФУНКЦИИ**


```sql

Задача 23
Пронумеровать персонажей каждого игрока (ROW_NUMBER).

SELECT 
    ROW_NUMBER() OVER (PARTITION BY username ORDER BY name) AS character_number,
    username,
    name
FROM 
	(select *
from players as t1
left join characters as t2
on t1.player_id = t2.player_id
)


Задача 24
Найти самого прокачанного персонажа у каждого игрока (RANK).

select 
RANK() OVER (PARTITION BY username ORDER BY level desc) AS character_number,
    username,
    name
from (select *
from players as t1
left join characters as t2
on t1.player_id = t2.player_id
)


Задача 25
Посчитать накопительное количество предметов у игрока.

select username, SUM(quantity) OVER (PARTITION BY t1.player_id ORDER BY quantity) AS cumulative_quantity
from players as t1
left join inventory as t2
on t1.player_id = t2.player_id



Задача 26
Найти разницу в уровне между персонажами игрока (LAG).

SELECT 
    username,
    level,
    LAG(level) OVER (ORDER BY level) AS prevs_level,
    level - LAG(level) OVER (ORDER BY level) AS level_diff
FROM players
ORDER BY level asc


Задача 27
Ранжировать игроков по суммарной стоимости инвентаря.

select username, rank() over (order by sum_value DESC) AS rank
from(
select username, sum(value) as sum_value
from players t1
left join inventory t2
on t1.player_id = t2.player_id
left join items t3
on t2.item_id = t3.item_id
where value is not null
group by username
order by sum_value desc
)

🟤 **7. ИГРОВАЯ АНАЛИТИКА**

```sql

Задача 28 (Wealth Distribution)
Распределение богатства игроков:

разбить игроков на группы по стоимости инвентаря
select username, sum_value, case 
        when sum_value > 30000 THEN 'богатый'
        when sum_value >= 20000 AND sum_value <= 30000 THEN 'средний'
        else 'бедный'
    end as category
from(
select username, sum(value) as sum_value
from players t1
left join inventory t2
on t1.player_id = t2.player_id
left join items t3
on t2.item_id = t3.item_id
where value is not null
group by username
order by sum_value desc
)


Задача 29 (Top Players)
Топ-5 игроков:

по уровню
по количеству персонажей
по богатству

WITH top_level AS (select username, 'топ по уровню' as segment
from players
order by level desc
limit 5
),
top_person as (
select username, count (name), 'топ по количеству персонажей' as segment
from players t1
left join characters t2
on t1.player_id = t2.player_id
group by username
order by count desc
limit 5
),
top_value as (
select username, sum(value) as sum_value, 'топ по богатству' as segment
from players t1
left join inventory t2
on t1.player_id = t2.player_id
left join items t3
on t2.item_id = t3.item_id
where value is not null
group by username
order by sum_value desc
limit 5)

select username,segment from top_level
union 
select username,segment from top_person
union
select username,segment from top_value
order by segment asc


Задача 30 (Character Analysis)
Самые популярные классы персонажей.

select class, count(class)
from players t1
left join characters t2
on t1.player_id = t2.player_id
group by class
order by count desc

⚫ **8. БИЗНЕС-КЕЙСЫ**

```sql

Задача 31 (Retention proxy)
Найти игроков с более чем 1 персонажем (условный retention).

select *
from (select username, count(name) as count_name
from players t1
left join characters t2
on t1.player_id = t2.player_id
group by username
)
where count_name > 1

Задача 32 (Engagement)
Определить самых активных игроков:

много персонажей
много предметов

WITH person_2 as (select *
from (select username, count(name) as count_name, 'Больше 1 персонажа' as flag_person
from players t1
left join characters t2
on t1.player_id = t2.player_id
group by username
)
where count_name > 1
),
object as(
select *
from(select username, count(name), 'Больше 5 предметов'
from players t1
left join inventory t2
on t1.player_id = t2.player_id
left join items t3
on t2.item_id = t3.item_id
where value is not null
group by username)
where count > 5
)

select * from person_2
union all
select * from object



Задача 33 (Whales)
Найти "китов" (игроков с максимальной стоимостью инвентаря).

SELECT username, balanc_invent AS max_balans
FROM (
    SELECT username, SUM(value) AS balanc_invent
    FROM players t1
    LEFT JOIN inventory t2 ON t1.player_id = t2.player_id
    LEFT JOIN items t3 ON t2.item_id = t3.item_id
    WHERE value IS NOT NULL
    GROUP BY username
) AS subquery
ORDER BY balanc_invent DESC
limit 1


Задача 34 (Balance issue)
Найти:

Самый выгодный квест по получению золота

select title, reward_gold
from quests
order by reward_gold desc
limit 1

🧠 **9. Витрина данных**

```sql

Задача 35
Собрать витрину игрока:

player_id
username
количество персонажей
средний уровень
суммарная стоимость инвентаря

select player_id, t1.username, count_pers, avg_level, price
from players t1
left join (select username, count(name) as count_pers, avg(level) as avg_level, sum(price_invent) as price
			from players t1
			left join characters t2
			on t1.player_id = t2.player_id
			left join (select player_id, sum(value) as price_invent
						from inventory t1 
						left join items t2
						on t1.item_id = t2.item_id
						group by player_id
						) t3
			on t1.player_id = t3.player_id
			group by username
			) t2
on t1.username = t2.username

Задача 36
Определить:

"новичков" (низкий уровень - ниже 50)
"ветеранов" (высокий уровень - больше 50)

select *, case
			when level >= 50 then 'ветеран'
			else 'новичок' end as category
from players



Задача 37
Сделать полный аналитический отчёт:

сколько всего игроков
средний уровень
топ-1 игрок по уровню
ник самого первый игрока по id

select count_players,avg_all_level, top_1_username_lvl, number_1_player
from(select '1' as num_join, count(username) as count_players
	from players) t1
left join (select '1' as num_join, avg(level) as avg_all_level
			from players) t2
on t1.num_join = t2.num_join
left join (select '1' as num_join, username as top_1_username_lvl
			from players
			where level = (select max(level)
			from players)) t3
on t2.num_join = t3.num_join
left join (select '1' as num_join, username as number_1_player
			from players
			where player_id = (select min(player_id)
								from players)) t4
on t3.num_join = t4.num_join

🟡 **10. Манипулирование данными**

```sql

Задача 38
Добавте нового игрока

INSERT INTO players (player_id, username, email, created_at, level)
VALUES (181, 'Banned_Cheter', 'chitak@gmail.ru', '18.06.2026', 999)


Задача 39
Удалите нарушителя (ID 181)

DELETE FROM players WHERE player_id = '181'

Задача 40
Исправьте почту на @yandex.ru у ID 118

UPDATE players
SET email = 'Cyber_Ninja@yandex.ru'
WHERE player_id = 118;